In [5]:
import torch
from torch import nn

In [6]:
#base model 
# =====================================================
# 🔹  Embeding_layer
# =====================================================
class Embeding_layer(nn.Module):
    def __init__(self,
                 vocab_size=256,
                 d_model=128,
                 max_len=100,
                 n_out=128,):
        super().__init__()

        # Byte embedding layer (0–255)
        self.embedding = nn.Embedding(vocab_size, d_model)

        # Positional embeddings
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.norm = nn.LayerNorm(d_model)
        # Final normalization
        
        self.projection = nn.Linear(in_features=d_model, out_features=n_out)
        self.final_norm = nn.LayerNorm(n_out)
        self.activation = nn.GELU()

    def forward(self, x):
        """
        x: (batch, seq_len) — byte indices [0–255]
        """
        batch_size, seq_len = x.size()
        device = x.device

        positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, -1)
        x = self.embedding(x) + self.pos_embedding(positions)

        x = self.projection(x)
        x = self.final_norm(x)
        x = self.activation(x)

        return x  # (B, L, d_model)
    
# 🔹 Residual Depthwise-Separable Multi-Kernel Block
class ResidualConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_sizes=[3, 5, 7], reduction=16):
        super().__init__()

        #mid_ch = max(in_ch // 16, 8)  # reduce dimension before heavy convs
        self.branches = nn.ModuleList()

        for k in kernel_sizes:
            branch = nn.Sequential(
                # (B) Reduce channels first
                #nn.Conv1d(in_ch, 1, kernel_size=1, bias=False),
                
                #nn.GroupNorm(num_groups=8, num_channels=mid_ch),
                #nn.GELU(),
                #nn.Dropout1d(0.25),

                # (A) Depthwise conv
                nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=k // 2, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=out_ch),
                nn.GELU(),
                nn.Dropout1d(0.25),

                # Pointwise to expand to out_ch
                #nn.Conv1d(mid_ch, out_ch, kernel_size=1, bias=False),
                #nn.GroupNorm(num_groups=8, num_channels=out_ch),
                #nn.GELU()
            )
            self.branches.append(branch)

        # Combine all kernel branches
        self.merge_conv = nn.Conv1d(out_ch * len(kernel_sizes), out_ch, kernel_size=1, bias=False)
        #self.merge_bn = nn.BatchNorm1d(out_ch)
        
        #self.se = SEBlock(out_ch, reduction)
        self.shortcut = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        # Parallel multi-kernel branches
        out = [branch(x) for branch in self.branches]
        out = torch.cat(out, dim=1)

        out = self.merge_conv(out)
        #out = self.merge_bn(out)
        #out = self.se(out)
        out += self.shortcut(x)
        return F.relu(out)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class URLBinaryCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, maxlen=100):
        super(URLBinaryCNN, self).__init__()
        self.maxlen = maxlen

        # Shared Layer (Global)
        self.shared_layer = nn.ModuleDict({
            "embeding":  Embeding_layer(vocab_size=vocab_size, max_len=maxlen, d_model=256, n_out=embed_dim),
            "conv": ResidualConvBlock(embed_dim, 64, kernel_sizes=[3,5,7],reduction=32),
            "conv2": ResidualConvBlock(64, 32, kernel_sizes=[3,5,7],reduction=16),
            #"conv3": ResidualConvBlockDW(64, 32, kernel_sizes=[3,5,7]),
            #"conv4": ResidualConvBlockDW(128, 64, kernel_sizes=[3,5,7]),
            #"conv5": ResidualConvBlockDW(64, 32, kernel_sizes=[3,5,7]),
            "bilstm": nn.LSTM(input_size=32, hidden_size=32,num_layers=1, batch_first=True, bidirectional=True),
            "layer_norm": nn.LayerNorm(32*2),
            "gelu1": nn.GELU(),
            "postconv": nn.Conv1d(32*2, 64*2, kernel_size=3, padding=1,stride=2),
            #"avg_pool1": nn.AvgPool1d(kernel_size=2),
            "fc1": nn.Linear(64 * maxlen , 64),
            "gelu2": nn.GELU(),
            "dropout1": nn.Dropout(0.5),
            
        })

        # Personalization Layer (Local)
        self.personal_layer = nn.ModuleDict({
            "fc2": nn.Linear(64, 64),
            "gelu3": nn.GELU(),
            "dropout2": nn.Dropout(0.5),
            "head": nn.Linear(64, 1)
        })

    def forward(self, x):
        # Shared layers
        x = self.shared_layer["embeding"](x)
        x = x.permute(0, 2, 1)

        x = self.shared_layer["conv"](x)
        x = self.shared_layer["conv2"](x)
        #x = self.shared_layer["conv3"](x)
        #x = self.shared_layer["max_pool"](x)
        x = x.permute(0, 2, 1)
        x, _ = self.shared_layer["bilstm"](x)
        x = self.shared_layer["layer_norm"](x)
        x = self.shared_layer["gelu1"](x)
        x = x.permute(0, 2, 1)
        x = self.shared_layer["postconv"](x)
        #x = self.shared_layer["avg_pool1"](x)
        x = x.permute(0, 2, 1)
        #x = x[:, -1, :]
        x = x.flatten(1)
        x = self.shared_layer["dropout1"](self.shared_layer["gelu2"](self.shared_layer["fc1"](x)))
        # Personalization head
        x = self.personal_layer["dropout2"](self.personal_layer["gelu3"](self.personal_layer["fc2"](x)))
        x = self.personal_layer["head"](x)
        return torch.sigmoid(x)
    
    def extract_features(self, x):
        """Return deep features before final FC layers."""
        # Transformer
        x = self.shared_layer["embeding"](x)
        x = x.permute(0, 2, 1)
        # Shared layers
        x = self.shared_layer["conv"](x)
        
        x = x.permute(0, 2, 1)
        x, _ = self.shared_layer["bilstm"](x)
        x = self.shared_layer["layer_norm"](x)
        #x = x[:, -1, :]
        x = x.flatten(1
                      )
        x = self.shared_layer["dropout1"](self.shared_layer["relu1"](self.shared_layer["fc1"](x)))
        return x




In [7]:
# ============================================================
# Character Encoding Setup
# ============================================================

# Allowed printable ASCII chars
ascii_chars = [chr(i) for i in range(32, 127)]

# Special control tokens
special_tokens = [
    '<PAD>', '<UNK>',
]

# Build vocab and mapping
vocab = special_tokens + ascii_chars
char2idx = {ch: i for i, ch in enumerate(vocab)}



def encode(text, max_len=60):
    indices = torch.full((max_len,), char2idx["<PAD>"], dtype=torch.long)
    text = text.lower()[:max_len]
    for i, c in enumerate(text):
        indices[i] = char2idx.get(c, char2idx["<UNK>"])

    return indices

In [8]:
model_folder = 'models'
#open model folder and read all torch models state dict like(kmac_Phishing_urls_best_loss_model.pt)
#creat model class URLBinaryCNN(nn.Module): for each model add them to dict with dict name slice of model name removeing _best_loss_model
#make a function to take url input and tokenis them using encode(text, max_len=128)
#pass through each model and print out put with model name

In [16]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from glob import glob




def load_all_models(model_folder="models"):
    model_dict = {}

    model_files = glob(os.path.join(model_folder, "*.pt"))

    for filepath in model_files:
        filename = os.path.basename(filepath)

        # remove _best_loss_model, _acc_model etc
        model_name = filename.replace("_best_loss_model.pt", "").replace(".pt", "")

        print(f"Loading model: {model_name}")

        model = URLBinaryCNN(vocab_size=97,maxlen=128)
        state = torch.load(filepath, map_location="cpu")
        model.load_state_dict(state)
        model.eval()

        model_dict[model_name] = model

    return model_dict


# ------------------------------------------------------
# 4. INFERENCE FUNCTION
# ------------------------------------------------------
def predict_url_with_all_models(url_input, model_dict):
    x = encode(url_input, max_len=128)
    x = x.reshape(1,x.shape[0])
    print(url_input)
    print("\n========= MODEL PREDICTIONS =========\n")
    for name, model in model_dict.items():
        with torch.no_grad():
            output = model(x).item()
        print(f"{name:30s}  -->  {output:.4f}")


# ------------------------------------------------------
# 5. USAGE
# ------------------------------------------------------
if __name__ == "__main__":
    models = load_all_models("models")
    while True:
        url = input("Enter URL to test: ")
        if url == 'bye':
            break
        predict_url_with_all_models(url, models)


Loading model: kmac_Phishing_urls
Loading model: maliscious_urls
Loading model: phishing-site-urls
Loading model: phiusiil-phishing
https://www.iledefrance-mobilites.fr/

========= MODEL PREDICTIONS =========

kmac_Phishing_urls              -->  0.2562
maliscious_urls                 -->  1.0000
phishing-site-urls              -->  0.7567
phiusiil-phishing               -->  0.0000
